# lsu-pria — Entrenamiento en Google Colab

Este notebook está pensado para correr la parte pesada del proyecto en **Google Colab**:
- entrenamiento CNN con GPU (`cuda`) cuando esté disponible,
- preparación y entrenamiento con **dataset propio**,
- o entrenamiento con **iLSU-T** a partir de archivos ya extraídos en Google Drive.

## Qué cubre
- instalación del repo y dependencias,
- verificación de GPU,
- entrenamiento `train-stack` para dataset propio,
- entrenamiento `ilsut-train` para iLSU-T,
- lectura rápida de métricas y artefactos generados.


## Requisitos recomendados

### En Colab
- `Runtime > Change runtime type > Hardware accelerator > GPU`
- preferentemente T4 / L4 / A100 si está disponible

### En Google Drive
#### Opción A — dataset propio
- uno o más `landmarks.csv`
- imágenes referenciadas en columnas como `img_raw_path` / `img_masked_path`

#### Opción B — iLSU-T
- carpeta ya extraída con estructura parecida a:
  - `.../source2/episodes/...`
  - `.../source2/whisperx/...`
- opcionalmente `source3/...`
- un JSON de keywords tipo `deliverables/ilsut_keywords.example.json`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# === Configuración general ===
REPO_URL = 'https://github.com/SEU_USUARIO_O_FORK/lsu-pria.git'
REPO_BRANCH = 'main'
WORKDIR = '/content/lsu-pria'

# Elegí: 'own' o 'ilsut'
DATASET_MODE = 'own'

# Carpeta base en Drive para resultados del notebook
DRIVE_OUT_ROOT = '/content/drive/MyDrive/lsu_pria_runs'

# === Dataset propio ===
# Podés usar --csv o varios --csvs
OWN_CSV = '/content/drive/MyDrive/lsu_pria_data/merged/landmarks.csv'
OWN_CSVS = []
OWN_GROUP_COL = 'subject_id'
OWN_CNN_IMAGE_COL = 'img_raw_path'
OWN_WORKDIR = f'{DRIVE_OUT_ROOT}/train_stack_colab'

# === iLSU-T ===
# root = carpeta donde ya están los archivos extraídos
ILSUT_ROOT = '/content/drive/MyDrive/iLSUT_extracted'
ILSUT_SOURCES = ['source2']
ILSUT_KEYWORDS_JSON = '/content/drive/MyDrive/lsu_pria_data/ilsut_keywords.json'
ILSUT_WORKDIR = f'{DRIVE_OUT_ROOT}/ilsut_train_colab'
ILSUT_PIPELINES = ['landmarks', 'cnn']
ILSUT_FPS = 2
ILSUT_MAX_PER_SEG = 12
ILSUT_USE_MASKED = False

# === Hiperparámetros CNN ===
CNN_EPOCHS = 5
CNN_DEVICE = 'cuda'  # en Colab debería ser cuda

# === Opciones extra ===
RUN_ABLATION = False  # en Colab normalmente no conviene, usa cámara
PREPROCESS = True
SKIN_MASK = False
CAMERA_LIKE = False


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

workdir = Path(WORKDIR)
if workdir.exists():
    print(f'Reutilizando repo existente: {workdir}')
else:
    !git clone -b {REPO_BRANCH} {REPO_URL} {WORKDIR}

%cd {WORKDIR}
!python3 -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .


In [ ]:
!nvidia-smi || true

import torch
print('torch.cuda.is_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_count =', torch.cuda.device_count())
    print('device_name =', torch.cuda.get_device_name(0))


## Opción A — Entrenamiento con dataset propio

Usa `train-stack`, que corre:
- `train_landmarks.py`
- `train_cnn.py`
- `dataset_stats.py`
- `eval_split.py`

Si `RUN_ABLATION=False`, evita la parte que depende de cámara.


In [ ]:
if DATASET_MODE == 'own':
    from pathlib import Path
    Path(OWN_WORKDIR).mkdir(parents=True, exist_ok=True)

    cmd = [
        'python3', 'lsupria.py', 'train-stack',
        '--work-dir', OWN_WORKDIR,
        '--cnn-image-col', OWN_CNN_IMAGE_COL,
        '--group-col', OWN_GROUP_COL,
        '--cnn-epochs', str(CNN_EPOCHS),
        '--cnn-device', CNN_DEVICE,
    ]

    if OWN_CSVS:
        cmd += ['--csvs', *OWN_CSVS]
    else:
        cmd += ['--csv', OWN_CSV]

    if not RUN_ABLATION:
        cmd += ['--skip-ablation']

    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('DATASET_MODE != own, se omite esta celda.')


## Opción B — Entrenamiento con iLSU-T

Usa `ilsut-train`, que en este repo puede:
- generar `episodes_generated.csv` desde los archivos extraídos,
- construir el manifest weak-labeled desde WhisperX,
- extraer ROI/landmarks,
- entrenar landmarks y/o CNN,
- correr `dataset_stats` y `eval_split`.

### Importante
- este flujo **no descarga** iLSU-T;
- asume que ya lo tenés extraído en Google Drive.


In [ ]:
if DATASET_MODE == 'ilsut':
    from pathlib import Path
    Path(ILSUT_WORKDIR).mkdir(parents=True, exist_ok=True)

    cmd = [
        'python3', 'lsupria.py', 'ilsut-train',
        '--root', ILSUT_ROOT,
        '--keywords', ILSUT_KEYWORDS_JSON,
        '--work-dir', ILSUT_WORKDIR,
        '--fps', str(ILSUT_FPS),
        '--max-per-seg', str(ILSUT_MAX_PER_SEG),
        '--cnn-epochs', str(CNN_EPOCHS),
        '--cnn-device', CNN_DEVICE,
        '--pipelines', *ILSUT_PIPELINES,
        '--sources', *ILSUT_SOURCES,
    ]

    if PREPROCESS:
        cmd.append('--preprocess')
    if SKIN_MASK:
        cmd.append('--skin-mask')
    if CAMERA_LIKE:
        cmd.append('--camera-like')
    if ILSUT_USE_MASKED:
        cmd.append('--cnn-use-masked')
    if not RUN_ABLATION:
        cmd.append('--skip-ablation')

    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('DATASET_MODE != ilsut, se omite esta celda.')


## Inspección rápida de resultados

Esta celda detecta automáticamente el `work_dir` según el modo elegido y muestra:
- archivos generados,
- `dataset_stats.md`,
- `eval_split.json`.


In [ ]:
from pathlib import Path
import json

work_dir = Path(OWN_WORKDIR if DATASET_MODE == 'own' else ILSUT_WORKDIR)
print('work_dir =', work_dir)

for p in sorted(work_dir.rglob('*')):
    if p.is_file():
        print(p)

stats_md = work_dir / 'results' / 'dataset_stats.md'
if stats_md.exists():
    print('\n===== dataset_stats.md =====\n')
    print(stats_md.read_text(encoding='utf-8'))

eval_json = work_dir / 'results' / 'eval_split.json'
if eval_json.exists():
    print('\n===== eval_split.json =====\n')
    print(json.dumps(json.loads(eval_json.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))


In [ ]:
from pathlib import Path
import shutil

work_dir = Path(OWN_WORKDIR if DATASET_MODE == 'own' else ILSUT_WORKDIR)
zip_base = str(work_dir)
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=str(work_dir))
print('ZIP generado:', zip_path)


## Sugerencias prácticas

- En Colab, para iterar rápido, empezá con:
  - `CNN_EPOCHS = 1` o `2`
  - `ILSUT_FPS = 2`
  - `ILSUT_MAX_PER_SEG = 12`
- Si el dataset es grande, evitá `RUN_ABLATION=True` porque esa parte está pensada para cámara local.
- Si usás iLSU-T, la calidad final depende muchísimo del JSON de keywords y de cuántas clases logre capturar WhisperX.
- Si querés usar este notebook con tu fork, cambiá `REPO_URL` arriba.
